# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/afifatanveerr/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
import pandas as pd

# Load dataset directly from GitHub
url = "https://raw.githubusercontent.com/afifatanveerr/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Dataset Loaded Successfully!")
print("Shape:", df.shape)

print("\nFirst 5 rows:")
display(df.head())

print("\nColumns:")
print(df.columns.tolist())

Dataset Loaded Successfully!
Shape: (30000, 44)

First 5 rows:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7



Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## My lane as an ML task (type)

I am working on the Content Refresh and SEO Optimization lane.

This problem is best framed as a binary classification task.

The objective is to predict whether a content page should be refreshed based on its SEO performance and engagement metrics.

The model predicts one of two classes:

1 = Needs Refresh
0 = Does Not Need Refresh

The predictions help editorial teams prioritize which pages should be updated first to improve organic traffic and search performance.

In [8]:
task_type = "Classification"

print("ML Task Type:", task_type)

ML Task Type: Classification


## Target or proxy

The dataset does not contain a direct label indicating whether a page should be refreshed.

Therefore, a proxy target must be created using observed SEO performance.

For this notebook, a page is considered to need refresh if it has:

very low impressions
very low clicks

This proxy is only for experimentation. In a real production system, the target would ideally come from historical editorial decisions or measured improvements after content updates.

In [9]:
# Create a simple proxy target
df["needs_refresh"] = (
    (df["impressions_90d"] < 20) &
    (df["clicks_90d"] < 5)
).astype(int)

print("Target Distribution:")
print(df["needs_refresh"].value_counts())

# Detect search position column
if "avg_position" in df.columns:
    position_col = "avg_position"
elif "average_position" in df.columns:
    position_col = "average_position"
else:
    position_col = None

target_columns = [
    "content_id",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "needs_refresh"
]

if position_col:
    target_columns.append(position_col)

display(df[target_columns].head())

Target Distribution:
needs_refresh
0    25183
1     4817
Name: count, dtype: int64


,content_id,impressions_90d,clicks_90d,pageviews_90d,needs_refresh,avg_position
0,content_304f48230142,3803,29,22,0,10.6
1,content_a1fb4e703a9e,15320,7,10,0,20.3
2,content_9aa793d4d895,12581,11,14,0,36.5
3,content_331d6c4de07b,11751,58,87,0,6.2
4,content_d99b7a2d90ca,19140,24,177,0,44.0


In [10]:
print(df["needs_refresh"].value_counts(normalize=True) * 100)

needs_refresh
0    83.943333
1    16.056667
Name: proportion, dtype: float64


## Success metric

The primary evaluation metric is F1 Score.

F1 Score is appropriate because the proxy target may be imbalanced.

It balances:

Precision (avoiding unnecessary refresh recommendations)

Recall (finding pages that truly need refresh)

Using Accuracy alone could be misleading if most pages belong to one class

In [11]:
metric = "F1 Score"

print("Chosen Evaluation Metric:", metric)

Chosen Evaluation Metric: F1 Score


## The unit of analysis
Each row in the dataset represents one content page.

Each page contains SEO, traffic, engagement, and content-related features measured over the last 90 days.

The ML model makes one prediction for each individual page.

In [12]:
columns = [
    "content_id",
    "search_volume",
    "competition",
    "word_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d"
]

if position_col:
    columns.append(position_col)

print("One row represents one content page.")

display(df[columns].head(10))

One row represents one content page.


,content_id,search_volume,competition,word_count,impressions_90d,clicks_90d,pageviews_90d,avg_position
0,content_304f48230142,10.0,0.67,3221.0,3803,29,22,10.6
1,content_a1fb4e703a9e,90.0,0.01,2481.0,15320,7,10,20.3
2,content_9aa793d4d895,0.0,0.00,3515.0,12581,11,14,36.5
3,content_331d6c4de07b,10.0,0.00,NaN,11751,58,87,6.2
4,content_d99b7a2d90ca,0.0,0.00,2803.0,19140,24,177,44.0
5,content_d4084a4bc775,720.0,1.00,3080.0,3970,1,4,8.5
6,content_9a34b442b552,0.0,0.00,3059.0,20,0,1,7.0
7,content_a63219c6e95a,590.0,0.44,NaN,1724,1,28,21.2
8,content_5e6c160719bc,0.0,0.00,3807.0,32574,29,128,46.0
9,content_c27558df2b0c,0.0,0.00,NaN,1240,2,4,4.9


## Why ML beats a fixed rule here

A fixed rule such as:

Refresh every page with fewer than 100 impressions.

does not capture the complexity of SEO performance.

For example:

Some pages naturally have low search demand.
Others may have high impressions but poor click-through rate.
Older content may perform differently from newer content.
Keyword competition also affects traffic.

Machine learning can learn patterns across many variables simultaneously and identify pages that are genuinely likely to benefit from a content refresh.

This produces more accurate and scalable recommendations than manually written rules.

In [13]:
features = [
    "search_volume",
    "competition",
    "word_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d"
]

optional_features = [
    "ctr",
    "engaged_sessions",
    "avg_position",
    "content_age_days"
]

for feature in optional_features:
    if feature in df.columns:
        features.append(feature)

print("Selected Features:")

for feature in features:
    print("-", feature)

display(df[features].head())

print("\nFeature Summary:")
display(df[features].describe())

Selected Features:
- search_volume
- competition
- word_count
- impressions_90d
- clicks_90d
- pageviews_90d
- ctr
- avg_position
- content_age_days


,search_volume,competition,word_count,impressions_90d,clicks_90d,pageviews_90d,ctr,avg_position,content_age_days
0,10.0,0.67,3221.0,3803,29,22,0.76,10.6,187
1,90.0,0.01,2481.0,15320,7,10,0.05,20.3,445
2,0.0,0.00,3515.0,12581,11,14,0.09,36.5,141
3,10.0,0.00,NaN,11751,58,87,0.49,6.2,463
4,0.0,0.00,2803.0,19140,24,177,0.13,44.0,263



Feature Summary:


,search_volume,competition,word_count,impressions_90d,clicks_90d,pageviews_90d,ctr,avg_position,content_age_days
count,27532.000000,27532.000000,22301.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.00000,30000.00000
mean,158.882391,0.146954,3107.760325,5200.366300,16.097333,49.942467,0.510733,16.34238,256.16780
std,1518.270825,0.285241,1452.382598,16838.019547,75.076958,152.101430,3.279162,15.21679,132.70793
min,0.000000,0.000000,8.000000,1.000000,0.000000,0.000000,0.000000,0.00000,90.00000
25%,0.000000,0.000000,2413.000000,81.000000,0.000000,2.000000,0.000000,6.20000,132.00000
50%,10.000000,0.000000,2877.000000,731.000000,1.000000,8.000000,0.070000,10.80000,236.00000
75%,20.000000,0.130000,3666.000000,3615.250000,7.000000,33.000000,0.290000,22.30000,333.00000
max,74000.000000,1.000000,9546.000000,517715.000000,4178.000000,5998.000000,100.000000,245.00000,564.00000


## Self-check

- [x] Every section above is completed with markdown and supporting code.
- [x] The notebook runs from top to bottom without errors.
- [x] No client names, URLs, or private queries are included.
- [x] My explanations use careful language such as observed, measured, proxy, and decision-support.
- [x] The notebook will be committed under `work/notebooks/` before submission.